# 04 - Challenger Model (LightGBM)

## Purpose

This notebook builds a challenger to the WoE scorecard from notebook 03: a gradient boosted tree model trained with LightGBM. Its purpose is to measure how much predictive power the scorecard gives up in exchange for its interpretability. It learns non-linear relationships and interactions between features directly from the prepared data, without manual binning, but still its predictions require an explanation method (SHAP will be used) to be interpreted.

## Inputs

- `data/processed/train_prepared.parquet` and `test_prepared.parquet` from notebook 02, with the same 96 features and the same excluded columns.

- `reports/scorecard_metrics.json`, the scorecard's test performance, which serves as the benchmark.

## Approach

1. Categorical features are converted to LightGBM's categorical type, with the categories defined on the training set.

2. The number of trees is chosen by 5-fold cross-validation with early stopping on the training set.

3. The final model is fitted on the full training set.

4. The model is evaluated once on the test set with the same metrics as the scorecard: Gini, KS, Brier score and calibration by decile.

5. SHAP values explain which features drive the predictions.

## Design decisions

- **No class reweighting.** Weighting defaulters more heavily during training would inflate the predicted probabilities. Since the PD is used directly in the IFRS 9 part, the natural default rate is preserved, as done in notebooks 01–03.

- **Same evaluation rule as notebook 03.** All choices are made on the training set, the test set is used once, so the comparison with the scorecard is made on identical, unseen applicants, avoiding leakage. 

## Setup and data loading

The prepared training set is loaded with the same features and exclusions as in notebook 03, together with the scorecard's test metrics as the benchmark. The test set is not loaded until the final evaluation.

In [1]:
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed = ROOT / "data" / "processed"
configs = ROOT / "configs"
reports = ROOT / "reports"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

train_df = pd.read_parquet(processed / "train_prepared.parquet")
with open(configs / "preparation_params.json") as f:
    prep_params = json.load(f)
with open(reports / "scorecard_metrics.json") as f:
    benchmark = json.load(f)

NON_FEATURE_COLS = list(prep_params["non_feature_cols"])
feature_cols = [c for c in train_df.columns if c not in NON_FEATURE_COLS]
X_train = train_df[feature_cols].copy()
y_train = train_df["TARGET"]
categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()

print(f"Training set: {X_train.shape[0]:,} rows, {len(feature_cols)} features "
      f"({len(categorical_cols)} categorical), default rate {y_train.mean():.4f}")
print(f"Benchmark (scorecard, test): Gini {benchmark['test_gini']:.4f}, KS {benchmark['test_ks']:.4f}")


Training set: 246,008 rows, 96 features (15 categorical), default rate 0.0807
Benchmark (scorecard, test): Gini 0.5001, KS 0.3695


### Categorical features 

LightGBM requires numeric input. The 15 categorical features are converted to pandas `category` type, which LightGBM handles natively by finding the best grouping of categories at each split, without imposing an artificial order. The list of categories is defined on the training set and the same definition is applied to the test set later on, so that both coincide. Categories not present in the training set become missing values.

In [2]:
# Categories of each feature defined on the training set. 

cat_dtypes = {
    col: pd.CategoricalDtype(categories=sorted(X_train[col].dropna().unique()))
    for col in categorical_cols
}

def to_lgb_categorical(frame, dtypes):
    frame = frame.copy()
    for col, dtype in dtypes.items():
        frame[col] = frame[col].astype(dtype)
    return frame

X_train = to_lgb_categorical(X_train, cat_dtypes)

print(X_train.dtypes.astype(str).value_counts().to_string())
print("\nCategories per features:")
print(pd.Series({c: len(d.categories) for c, d in cat_dtypes.items()}).sort_values(ascending=False).to_string())


float64     44
int64       37
category    15

Categories per features:
ORGANIZATION_TYPE             58
OCCUPATION_TYPE               18
NAME_INCOME_TYPE               8
WALLSMATERIAL_MODE             7
WEEKDAY_APPR_PROCESS_START     7
NAME_TYPE_SUITE                7
NAME_FAMILY_STATUS             6
NAME_HOUSING_TYPE              6
NAME_EDUCATION_TYPE            5
FONDKAPREMONT_MODE             4
HOUSETYPE_MODE                 3
FLAG_OWN_CAR                   2
NAME_CONTRACT_TYPE             2
FLAG_OWN_REALTY                2
EMERGENCYSTATE_MODE            2


**What I see:** All 15 categorical features were converted to the `category` type, with categories defined on the training set, together with 44 float and 37 integer features this gives the 96 features of notebook 03. The category count ranges from 2 for the binary flags to 58 for `ORGANIZATION_TYPE`, which includes the `XNA` code for applicants without an employer.

**Decision:** The same category definitions will be applied to the test set in the final evaluation, so that each category maps to the same internal code in both sets.

## Gradient boosting

Gradient boosting builds a prediction from many small decision trees, added one at a time. It starts from a simple prediction, the average log-odds of default, and then repeats two steps:

1. For every applicant, the gradient of the loss function is computed. It describes in which direction and by how much the current prediction is wrong. The loss is the log loss (binary cross-entropy measuring how close the model is to the observed outcomes), the same loss is minimised by logistic regression.

2. A new small tree is then fitted to these gradients, so that it learns to correct the current errors. Its output is added to the prediction, scaled by a **learning rate** (0.05 here), so that no single tree has too much influence.

The final prediction is the sum of all trees on the log-odds scale, converted into a probability of default. Unlike a logistic-regression, the model does not assign one weight per feature: each tree adjusts the prediction of each applicant through the leaf they fall into, and the importance of a feature only emerges from how the trees use it. This is why SHAP is needed to explain the model.

Down below is a comparison between the two models:

| | WoE scorecard | LightGBM |
|---|---|---|
| Input | WoE values from manual binning | Prepared features, unbinned |
| Non-linear relationships | Captured by the binning | Learned by the splits |
| Interactions between features | Not captured | Captured by consecutive splits |
| Missing values | Own WoE bin | Each split learns where to send them |
| Explanation | Points per bin | Requires SHAP |

## Cross-validated training

The model is trained with 5-fold stratified cross-validation on the training set. In each fold, trees are added as long as the AUC on the held-out fold improves, training stops after 100 trees without improvement (early stopping). The number of trees with the best average AUC across folds is used for the final model.

The hyperparameters are moderate and conservative, commonly used starting values rather than the result of a search: a learning rate of 0.05, at most 31 leaves per tree, at least 200 applicants per leaf, and random subsampling of 80% of rows and features per tree. No class reweighting is applied.

In [3]:
from sklearn.model_selection import StratifiedKFold

params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_child_samples": 200,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "seed": 42,
    "verbose": -1,
}

train_set = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_cols, free_raw_data=False)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = lgb.cv(
    params, train_set,
    num_boost_round=3000,
    folds=cv,
    callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=200)],
)

best_n_trees = len(cv_results["valid auc-mean"])
cv_auc = cv_results["valid auc-mean"][-1]
cv_auc_std = cv_results["valid auc-stdv"][-1]
print(f"\nBest number of trees: {best_n_trees}")
print(f"Cross-validated Gini: {2 * cv_auc - 1:.4f} (std {2 * cv_auc_std:.4f})")
print(f"Scorecard cross-validated Gini: {benchmark['cv_gini_mean']:.4f}")

c:\dev\credit-risk-pd-model\.venv\Lib\site-packages\sklearn\model_selection\_split.py:884: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[200]	valid's auc: 0.760854 + 0.00354824
[400]	valid's auc: 0.761462 + 0.0034389
Early stopping, best iteration is:
[348]	valid's auc: 0.761647 + 0.00334859

Best number of trees: 348
Cross-validated Gini: 0.5233 (std 0.0067)
Scorecard cross-validated Gini: 0.4955


**What I see:** Early stopping selects 348 trees. The cross-validated AUC is 0.7616, a Gini of 0.5233 (std 0.0067), against 0.4955 (std 0.0062) for the scorecard. Most of the improvement in AUC occurs within the first 200 trees. A warning that StratifiedKFold ignores the `groups` parameter appears solely due to the interaction between LightGBM and scikit-learn and does not affect the results.

**Interpretation:** The LightGBM method improves the cross-validated Gini by about 2.8 points, roughly four times the variation between folds, so the difference is unlikely to be noise. Possible sources are interactions between features, finer non-linear patterns than six bins allow, and the combined contributions of features removed from the scorecard, these are examined with SHAP. Both cross-validated estimates are slightly optimistic for different reasons, so the decisive comparison is made on the test set.

**Decision:** The final model is trained on the full training set with 348 trees. 

## Final model

The final model is trained on the full training set with the hyperparameters above and the 348 trees selected by cross-validation. As a diagnostic, its Gini on the training data itself is compared with the cross-validated one, the gap shall show how much the model fits patterns specific to the training set sample.

In [4]:
from sklearn.metrics import roc_auc_score

final_model = lgb.train(params, train_set, num_boost_round=best_n_trees)

p_train = final_model.predict(X_train)
gini_train = 2 * roc_auc_score(y_train, p_train) - 1
gini_cv = 2 * cv_auc - 1

print(f"Trees: {final_model.num_trees()}")
print(f"Gini on the training data itself: {gini_train:.4f}")
print(f"Cross-validated Gini:              {gini_cv:.4f}")
print(f"Gap:                               {gini_train - gini_cv:.4f}")

Trees: 348
Gini on the training data itself: 0.6532
Cross-validated Gini:              0.5233
Gap:                               0.1299


**What I see:** The final model contains 348 trees. Its Gini on the training data itself is 0.6532, against a cross-validated Gini of 0.5233, a gap of 0.13.

**Interpretation:** The gap measures how much the model fits patterns specific to the training sample. With 348 trees of up to 31 leaves, the model has enough flexibility to memorise part of the training data, which does not carry over to new applicants (overfitting). The training Gini therefore overstates performance and is not used for evaluation. The cross-validated Gini still exceeds the scorecard's, so the model captures additional real signal despite this optimistic memorisation. A scorecard with 19 WoE features has far less capacity to memorise, which shows the trade-off between flexibility and robustness. Stronger regularisation could narrow the gap and is noted as a possible extension, the calibration on the test set will show whether the memorisation makes the predicted probabilities overconfident.

**Decision:** The model is evaluated once on the test set.

## Final evaluation on the test set

The test set is used here for the first and only time. The categorical features are converted with the category definitions learned on the training set, and the model is evaluated with the same metrics as the scorecard: Gini, KS, Brier score and calibration by decile. Since both models are evaluated on the same unseen applicants, the comparison isolates their differences. 

In [5]:
from sklearn.metrics import roc_curve, brier_score_loss

# Using the test set.
test_df = pd.read_parquet(processed / "test_prepared.parquet")
X_test = to_lgb_categorical(test_df[feature_cols], cat_dtypes)
y_test = test_df["TARGET"]
assert (X_test.dtypes == X_train.dtypes).all(), "Train and test types differ"

#Categories in the test set but unseen in the training become missing.
unseen = {c: int((test_df[c].notna() & X_test[c].isna()).sum()) for c in categorical_cols}
unseen = {c: n for c, n in unseen.items() if n > 0}
print("Test values with categories unseen in training:", unseen if unseen else "none")

p_test = final_model.predict(X_test)
gini_test = 2 * roc_auc_score(y_test, p_test) - 1
fpr, tpr, _ = roc_curve(y_test, p_test) #True and False positive rate
ks_test = np.max(tpr - fpr)
brier_test = brier_score_loss(y_test, p_test)

comparison = pd.DataFrame (
    {"Scorecard": [benchmark["test_gini"], benchmark["test_ks"], benchmark["test_brier"]], 
    "LightGBM": [gini_test, ks_test, brier_test]}, index=["Gini", "KS", "Brier"],
)
comparison["Difference"] = comparison["LightGBM"] - comparison["Scorecard"]
print(f"\nBrier baseline: {benchmark['brier_baseline']:.4f}")
comparison.round(4)

Test values with categories unseen in training: none

Brier baseline: 0.0742


,Scorecard,LightGBM,Difference
Gini,0.5001,0.5414,0.0413
KS,0.3695,0.4018,0.0324
Brier,0.0685,0.0672,-0.0013


In [6]:
calib = pd.DataFrame({"predicted_pd": p_test, "default": y_test.values})
calib["decile"] = pd.qcut(calib["predicted_pd"], q=10, labels=False) + 1

calibration_table = calib.groupby("decile").agg(
    n=("default", "size"),
    mean_predicted=("predicted_pd", "mean"),
    observed_rate=("default", "mean"),
).round(4)

print(f"Mean predicted PD: {p_test.mean():.4f} vs observed default rate: {y_test.mean():.4f}\n")
calibration_table

Mean predicted PD: 0.0802 vs observed default rate: 0.0807



,n,mean_predicted,observed_rate
decile,,,
1,6151,0.0133,0.0089
2,6150,0.0210,0.0190
3,6150,0.0279,0.0237
4,6150,0.0357,0.0345
5,6151,0.0449,0.0460
6,6150,0.0569,0.0595
7,6150,0.0738,0.0774
8,6150,0.0992,0.1050
9,6150,0.1452,0.1489


**What I see:** No test values had categories unseen in training. On the test set, LightGBM outperforms the scorecard on all three metrics: Gini 0.5414 against 0.5001 (+0.041), KS 0.4018 against 0.3695 (+0.032), and Brier score 0.0672 against 0.0685. Observed default rates range from 0.89% in the safest decile to 28.4% in the riskiest, against 1.5% to 25.9% for the scorecard. Averaged over the deciles, the predicted and observed rates agree (8.02% against 8.07%), and the largest absolute gap is 0.58 percentage points. However, the model over predicts in the four lowest deciles and inversely under predicts in deciles 5 to 9: in the lowest decile it predicts 1.33%  against an observed 0.89%, about 3.7 standard errors.

**Interpretation:** The LightGBM ranks the applicants more sharply, separating the safest and riskiest deciles by a factor of about 32 against 17 for the scorecard. Its calibration is accurate overall but shows a systematic pattern, it being that predictions for the safest applicants are not low enough, being here the opposite of overconfidence. One possible motive is regularisation, pulling leaf values towards the average. For IFRS 9, the resulting over estimation of PD for low-risk applicants is conservative and small in absolute terms, but the PDs should not be used without noting it. The Gini difference is likely to be statistically meaningful, since both models are evaluated on the same applicants, a formal test is deferred to the validation notebook. 

**Decision:** LightGBM is retained as the challenger with its current predictions. Recalibration, fitted on cross-validated training predictions, is noted as an extension for notebook 06. Next, the model's behaviour is examined with SHAP. 

## Explaining the model with SHAP

Unlike the scorecard, the LightGBM model has no table of points, so its predictions are explained with SHAP (SHapley Additive exPlanations). Based on Shapley values from cooperative game theory, SHAP divides the difference between an applicant's prediction and the average prediction among the features, such that

$$
\text{base value} + \sum_{j} \text{SHAP}_j = \text{predicted log-odds of default}.
$$

A positive SHAP value means a feature raises the applicant's predicted risk, a negative value lowers it. The values are computed exactly with TreeSHAP, which is built into LightGBM and handles categorical splits. They are on the same log-odds scale as the scorecard, so they can be expressed as scorecard equivalent points, $-\text{SHAP} \times \text{Factor}$, with Factor
= 20 / ln 2 as in notebook 03.

The explanations are computed on the test set. This changes nothing about the model, it shows how the model behaves on unseen applicants. 

In [7]:
# TreeSHAP values.
contrib = final_model.predict(X_test, pred_contrib=True)
shap_values = pd.DataFrame(contrib[:, :-1], columns=feature_cols, index=X_test.index)
base_value = contrib[0, -1]

# Check: base value + contributions reproduce the model's log-odds exactly.
raw_log_odds = final_model.predict(X_test, raw_score=True)
assert np.allclose(shap_values.sum(axis=1) + base_value, raw_log_odds), "SHAP values do not add up"

print(f"SHAP values: {shap_values.shape[0]:,} applicants × {shap_values.shape[1]} features")
print(f"Base value (average log-odds of default): {base_value:.4f}")

SHAP values: 61,503 applicants × 96 features
Base value (average log-odds of default): -2.8339


In [8]:
with open(configs / "scorecard_params.json") as f:
    scorecard_features = json.load(f)["features"]

importance = shap_values.abs().mean().sort_values(ascending=False)
importance_table = pd.DataFrame ({
    "mean_abs_shap": importance.round(4),
    "share_%": (importance / importance.sum() * 100).round(1),
    "in_scorecard": importance.index.isin(scorecard_features),
})

share_scorecard = importance[importance.index.isin(scorecard_features)].sum() / importance.sum()
print(f"Share of total importance from the 19 scorecard features: {share_scorecard:.1%}")
importance_table.head(20)

Share of total importance from the 19 scorecard features: 75.8%


,mean_abs_shap,share_%,in_scorecard
EXT_SOURCE_3,0.3484,14.8,True
EXT_SOURCE_2,0.3100,13.2,True
EXT_SOURCE_1,0.1662,7.1,True
ANNUITY_CREDIT_RATIO,0.1462,6.2,True
CREDIT_GOODS_RATIO,0.1194,5.1,True
YEARS_EMPLOYED,0.0989,4.2,True
ORGANIZATION_TYPE,0.0893,3.8,True
OWN_CAR_AGE,0.0726,3.1,True
AMT_ANNUITY,0.0680,2.9,True
OCCUPATION_TYPE,0.0676,2.9,True


In [9]:
# For the most important numerical features: does higher value mean higher or lower risk?
top_numeric = [f for f in importance.index[:10] if f not in categorical_cols]
direction = pd.Series({
    f: X_test[f].corr(shap_values[f], method="spearman") for f in top_numeric
}).round(3)
direction

EXT_SOURCE_3           -0.990
EXT_SOURCE_2           -0.991
EXT_SOURCE_1           -0.973
ANNUITY_CREDIT_RATIO    0.243
CREDIT_GOODS_RATIO      0.907
YEARS_EMPLOYED         -0.961
OWN_CAR_AGE             0.805
AMT_ANNUITY             0.823
dtype: float64

In [10]:
factor = 20 / np.log(2)

# The applicant with the highest predicted PD in the test set.
idx = X_test.index[np.argmax(p_test)]
local = pd.DataFrame({
    "value": X_test.loc[idx].astype(str),
    "shap": shap_values.loc[idx], 
})
local["points_equivalent"] = (-local["shap"] * factor).round(1)
local = local.reindex(local["shap"].abs().sort_values(ascending=False).index)

print(f"Predicted PD: {p_test.max():.3f} (average: {p_test.mean():.3f})")
local.head(8).round(4)

Predicted PD: 0.733 (average: 0.080)


,value,shap,points_equivalent
EXT_SOURCE_3,0.0463097464917085,1.4883,-42.9
ANNUITY_CREDIT_RATIO,0.0792797583999696,0.4809,-13.9
YEARS_EMPLOYED,1.2054794520547945,0.2240,-6.5
EXT_SOURCE_2,0.4164070503730191,0.1938,-5.6
OCCUPATION_TYPE,Low-skill Laborers,0.1604,-4.6
YEARS_BIRTH,36.35616438356164,0.1510,-4.4
CREDIT_GOODS_RATIO,1.2243953488372092,0.1482,-4.3
EXT_SOURCE_1,NaN,0.1399,-4.0


**What I see:** The SHAP values add up exactly to the model's predictions for all 61,503 test applicants. The base value is -2.834, the average log-odds across applicants, which corresponds to a PD of about 5.6%. The 19 scorecard features account for 75.8% of the total importance, the eleven most important features are all scorecard features, led by `EXT_SOURCE_3` (14.8%), `EXT_SOURCE_2`(13.2%) and `EXT_SOURCE_1` (7.1%). Among the features outside the scorecard, `YEARS_BIRTH` (2.7%), `FLAG_WORK_PHONE`, `REGION_POPULATION_RELATIVE`, `AMT_REQ_CREDIT_BUREAU_QRT` and `DEF_30_CNT_SOCIAL_CIRCLE` appear in the top 20. The external scores have almost perfectly negative directional relationships with their SHAP values (Spearman -0.97 to -0.99), `CREDIT_GOODS_RATIO` a strongly positive one (+0.91), and `ANNUITY_CREDIT_RATIO` a weak one (+0.24). For the highest-risk applicant (PD 73.3%), a very low `EXT_SOURCE_3` contributes the equivalent of -42.9 scorecard points, and a missing `EXT_SOURCE_1` raises the risk.

**Interpretation:** The LightGBM's advantage comes mainly from using the scorecard's own features more freely, through finer non-linear patterns and interactions, and only partly from additional features. Two methodologically different models identify the same key predictors, which supports further the robustness of these findings. All examined directions are consistent with notebook 01 and 02, including the non-monotonic edge cases of `ANNUITY_CREDIT_RATIO` and the informative missingness of `EXT_SOURCE_1`. However, here ages are directly used, which the scorecard does not, and relies on defaults within the applicant's social circle, a feature that is predictive but ethically contestable. SHAP makes individual predictions explainable as reason codes, which is a precondition for using such a model under for reference the GDPR Article 22 in Europe.

**Decision:** The use of age and social circle features is carried into the fairness analysis of notebook 06.

## Saving the challenger

The test metrics, the comparison with the scorecard, the calibration table and the SHAP importances are written to `reports/`. The hyperparameters, number of trees and category definitions are written to `configs/challenger_params.json`. The model is saved in LightGBM's native text format to `models/`, which is excluded from the version control. 

In [11]:
models = ROOT / "models"
models.mkdir(exist_ok=True)

# Human readable outputs. (committed)
comparison.round(4).to_csv(reports / "model_comparison_test.csv")
calibration_table.to_csv(reports / "challenger_calibration_test.csv")
importance_table.to_csv(reports / "challenger_shap_importance.csv")

challenger_metrics = {
    "n_trees": int(best_n_trees),
    "cv_gini_mean": float(2 * cv_auc - 1),
    "cv_gini_std": float(2 * cv_auc_std),
    "train_gini": float(gini_train),
    "test_gini": float(gini_test),
    "test_ks": float(ks_test),
    "test_brier": float(brier_test),
    "shap_share_scorecard_features": float(share_scorecard),
}
with open(reports / "challenger_metrics.json", "w") as f:
    json.dump(challenger_metrics, f, indent=2)

# Settings for model's reproduction. (committed)

challenger_params = {
    "lightgbm_params": params,
    "n_trees": int(best_n_trees),
    "categories": {col: [str(v) for v in d.categories] for col, d in cat_dtypes.items()},
}
with open(configs / "challenger_params.json", "w") as f:
    json.dump(challenger_params, f, indent=2)


# Model in LightGBM's native text format. (not committed)
model_path = models / "challenger_lgbm.txt"
final_model.save_model(str(model_path))

reloaded = lgb.Booster(model_file=str(model_path))
assert np.allclose(reloaded.predict(X_test), p_test), "Reloaded model differs"
print("Saved: comparisons, calibration, SHAP importance, metrics, parameters and model.")


Saved: comparisons, calibration, SHAP importance, metrics, parameters and model.


## Summary and handoff to notebook 05

This notebook trained a LightGBM model on the same 96 features and the same train/test split as the scorecard. Five fold cross-validation with early stopping selected 348 trees, with a cross-validated Gini of 0.523 against 0.496 for the scorecard.

**Results on the test set:** Gini 0.541 against 0.500 for the scorecard, KS 0.402 against 0.370, and Brier score 0.0672 against 0.0685. The LightGBM ranks applicants more sharply, but over-predicts the PD of the safest applicants. SHAP attributes 76% of its importance to the scorecard's own 19 features, so most of the gain comes from using the same information more flexibly.

**Trade-off:** the challenger gains about four Gini points at the cost of transparency. Its predictions need SHAP to be explained, and it uses age and social-circle features that the scorecard does not.

**Notebook 05 receives:** the scorecard as the primary model for the IFRS 9 expected credit loss, with the challenger's PDs available for comparison.

**Open points for notebook 06:**
- A formal test of the Gini difference.
- Recalibration of the challenger, fitted on cross-validated training predictions.
- Fairness analysis by gender and age, including the challenger's direct use of age and of social-circle features.
